# P4 — Ensemble Anomaly Detection V2
4-component weighted ensemble: Autoencoder(0.4) + IF(0.3) + Z-score(0.2) + ClusterDev(0.1)
Graduate if ensemble ROC-AUC > Isolation Forest baseline ROC-AUC.

In [1]:
import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, joblib, warnings
warnings.filterwarnings('ignore')
ROOT = os.path.abspath('..')
if ROOT not in sys.path: sys.path.insert(0, ROOT)
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import MinMaxScaler, StandardScaler
try:
    import tensorflow as tf
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense
    from tensorflow.keras.callbacks import EarlyStopping
    HAS_TF = True; print(f'TensorFlow {tf.__version__}')
except ImportError:
    HAS_TF = False; print('TensorFlow not found. pip install tensorflow')
sns.set_theme(style='whitegrid'); print('Setup complete.')

I0000 00:00:1778701347.660386    2042 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778701347.661249    2042 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778701349.538537    2042 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778701349.539210    2042 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow 2.21.0
Setup complete.


In [2]:
CSV_PATH = os.path.join(ROOT, 'ai4i2020.csv')
df = pd.read_csv(CSV_PATH)
SENSOR_COLS = ['Air temperature [K]','Process temperature [K]',
               'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
X = df[SENSOR_COLS].values
y_true = df['Machine failure'].values
X_healthy = X[y_true == 0]
MODELS_DIR = os.path.join(ROOT, 'app', 'backend', 'modules', 'ml', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Dataset: {df.shape}, Healthy: {len(X_healthy)}')

Dataset: (10000, 14), Healthy: 9661


## Step 1 — Baseline: Isolation Forest

In [3]:
iso_path = os.path.join(MODELS_DIR, 'ml_model_p4_anomaly.pkl')
if os.path.exists(iso_path):
    existing = joblib.load(iso_path)
    iso_model = existing.get('model', existing) if isinstance(existing, dict) else existing
    print('Loaded existing Isolation Forest.')
else:
    iso_model = IsolationForest(contamination=0.034, random_state=42, n_jobs=-1)
    iso_model.fit(X_healthy); print('Trained new Isolation Forest.')
S_IF_raw = -iso_model.decision_function(X)
baseline_roc = roc_auc_score(y_true, S_IF_raw)
print(f'Baseline IF ROC-AUC: {baseline_roc:.4f}')

Loaded existing Isolation Forest.


ValueError: X has 5 features, but IsolationForest is expecting 7 features as input.

## Step 2 — Autoencoder (trained on healthy only)

In [ ]:
if HAS_TF:
    ae_scaler = StandardScaler()
    Xh_sc = ae_scaler.fit_transform(X_healthy)
    Xa_sc = ae_scaler.transform(X)
    n = X.shape[1]
    inp = Input(shape=(n,))
    enc = Dense(16,'relu')(inp); enc = Dense(8,'relu')(enc); btn = Dense(4,'relu')(enc)
    dec = Dense(8,'relu')(btn);  dec = Dense(16,'relu')(dec); out = Dense(n)(dec)
    autoencoder = Model(inp, out); autoencoder.compile('adam','mse')
    autoencoder.fit(Xh_sc, Xh_sc, epochs=100, batch_size=64, validation_split=0.1,
                   callbacks=[EarlyStopping(patience=10,restore_best_weights=True)], verbose=1)
    Xr = autoencoder.predict(Xa_sc, verbose=0)
    S_AE_raw = np.mean((Xa_sc - Xr)**2, axis=1)
    ae_roc = roc_auc_score(y_true, S_AE_raw)
    print(f'Autoencoder ROC-AUC: {ae_roc:.4f}')
else:
    S_AE_raw = np.zeros(len(X)); ae_roc = 0.5; print('AE skipped.')

## Step 3 — Z-Score + Cluster Deviation

In [ ]:
S_zscore_raw = np.max(np.abs(stats.zscore(X, axis=0)), axis=1)
zscore_roc = roc_auc_score(y_true, S_zscore_raw)
print(f'Z-score ROC-AUC: {zscore_roc:.4f}')
pipeline_path = os.path.join(MODELS_DIR, 'feature_pipeline_v3.pkl')
if os.path.exists(pipeline_path):
    pipeline = joblib.load(pipeline_path)
    X_v3 = pipeline.transform(df[SENSOR_COLS])
    S_cluster_raw = X_v3['cluster_distance'].values if 'cluster_distance' in X_v3.columns else np.zeros(len(X))
    cluster_roc = roc_auc_score(y_true, S_cluster_raw) if S_cluster_raw.any() else 0.5
    print(f'Cluster deviation ROC-AUC: {cluster_roc:.4f}')
else:
    S_cluster_raw = np.zeros(len(X)); cluster_roc = 0.5
    print('Feature pipeline not found. Run p3_lstm_hybrid_rul.ipynb first.')

## Step 4 — Weighted Ensemble + Graduate Decision

In [ ]:
def normalize(s): return MinMaxScaler().fit_transform(s.reshape(-1,1)).flatten()
W_AE,W_IF,W_Z,W_C = 0.40,0.30,0.20,0.10
S_final = W_AE*normalize(S_AE_raw)+W_IF*normalize(S_IF_raw)+W_Z*normalize(S_zscore_raw)+W_C*normalize(S_cluster_raw)
ensemble_roc = roc_auc_score(y_true, S_final)
print(f'Ensemble ROC-AUC: {ensemble_roc:.4f}  |  Baseline IF: {baseline_roc:.4f}')
should_graduate = ensemble_roc > baseline_roc
print(f'GRADUATE: {"YES" if should_graduate else "NO"}')
fig, ax = plt.subplots(figsize=(8,6))
for s, lbl in [(S_final,f'Ensemble({ensemble_roc:.3f})'),(S_IF_raw,f'IF({baseline_roc:.3f})'),(S_AE_raw,f'AE({ae_roc:.3f})')]:
    fpr,tpr,_ = roc_curve(y_true,s); ax.plot(fpr,tpr,label=lbl,linewidth=2)
ax.plot([0,1],[0,1],'k--'); ax.set_title('P4 V2 ROC'); ax.legend(); plt.show()

In [ ]:
if should_graduate:
    data = {'iso_model':iso_model,'weights':{'ae':W_AE,'if':W_IF,'zscore':W_Z,'cluster':W_C},
            'thresholds':{'ae_max':float(S_AE_raw.max()),'if_min':float(S_IF_raw.min()),
                          'if_max':float(S_IF_raw.max()),'zscore_max':float(S_zscore_raw.max()),
                          'cluster_max':float(S_cluster_raw.max())},
            'metrics':{'ensemble_roc_auc':round(ensemble_roc,4),'baseline_roc_auc':round(baseline_roc,4)},
            'type':'anomaly_ensemble_v2'}
    if HAS_TF:
        ae_p = os.path.join(MODELS_DIR,'autoencoder_p4.keras')
        autoencoder.save(ae_p); data['autoencoder_path']=ae_p; data['ae_scaler']=ae_scaler
    out = os.path.join(MODELS_DIR,'ml_model_p4_anomaly_v2.pkl')
    joblib.dump(data,out); print(f'Saved: {out}')
else:
    print('Not saved. Keep original IF.')